# Judge Audit — Gemini 2.5 Flash

For each row in the pooled results CSV, Gemini checks:
1. Is the adapted puzzle structurally valid (context, clues, question, A-E choices present)
2. Did the judge reach a final decision or bail out early
3. Did the judge cite genuine equivalence issues or only formatting artifacts
4. What was the judge's final decision

Outputs a CSV with audit verdicts for filtering the final benchmark pool.

## 1. Install & Imports

In [ ]:
!pip install -q google-genai pandas openpyxl tqdm

In [ ]:
import re
import os
import json
import time
import pandas as pd
from tqdm import tqdm
from google import genai
from google.genai import types

## 2. Config

In [ ]:
# ── API ───────────────────────────────────────────────────────────────────────
GEMINI_API_KEY = 'YOUR_GEMINI_API_KEY'   # replace or load from env
MODEL_ID       = 'gemini-2.5-flash-preview-05-20'

# ── Paths ─────────────────────────────────────────────────────────────────────
# Input: your pooled results CSV containing adapted puzzles + judge verdicts
# Must have columns: Puzzle_Number, Question_Number, final_adapted_puzzle,
#                    final_judge_verdict, final_judge_think_trace, final_outcome
DATA_PATH  = 'pooled_results.csv'
WORKDIR    = './outputs'
os.makedirs(WORKDIR, exist_ok=True)
AUDIT_OUT  = os.path.join(WORKDIR, 'judge_audit.csv')

# ── Generation ────────────────────────────────────────────────────────────────
SAVE_EVERY        = 10
MAX_OUTPUT_TOKENS = 1024    # audit response is short — just JSON
THINKING_BUDGET   = 0       # no thinking needed — this is pattern recognition

# ── Filter: which rows to audit ───────────────────────────────────────────────
# Options: 'all', 'judge_failed', 'not_equivalent', 'equivalent'
# Recommended: audit 'equivalent' rows for the final benchmark pool
# and 'judge_failed' to rescue valid puzzles the judge couldn't evaluate
AUDIT_OUTCOMES = ['equivalent', 'judge_failed', 'not_equivalent']

## 3. Init Client

In [ ]:
client = genai.Client(api_key=GEMINI_API_KEY)
print(f'Client ready. Model: {MODEL_ID}')

## 4. Load Data

In [ ]:
df = pd.read_csv(DATA_PATH)

required = [
    'Puzzle_Number', 'Question_Number', 'final_adapted_puzzle',
    'final_judge_verdict', 'final_judge_think_trace', 'final_outcome',
]
for col in required:
    assert col in df.columns, f'Missing column: {col}'

# Filter to only the outcome types we want to audit
audit_df = df[df['final_outcome'].isin(AUDIT_OUTCOMES)].copy()
print(f'Total rows      : {len(df)}')
print(f'Rows to audit   : {len(audit_df)}')
print(f'Outcome counts  :')
print(audit_df['final_outcome'].value_counts().to_string())
audit_df.head()

## 5. Helpers

In [ ]:
def save_results(records: list, path: str):
    pd.DataFrame(records).to_csv(path, index=False)


def load_completed(path: str, key_cols: list):
    if not os.path.exists(path):
        return [], set()
    done_df   = pd.read_csv(path)
    done_keys = set(zip(*[done_df[c] for c in key_cols]))
    records   = done_df.to_dict('records')
    print(f'  Resuming: {len(records)} rows already in {os.path.basename(path)}')
    return records, done_keys


def generate(system_prompt: str, user_prompt: str) -> tuple:
    """
    Returns (raw_text, elapsed_minutes).
    Thinking disabled (THINKING_BUDGET=0) — audit is pattern recognition,
    not reasoning. Retries on 503/429 with exponential backoff.
    """
    max_retries = 5
    wait = 30

    for attempt in range(max_retries):
        try:
            t0 = time.time()
            response = client.models.generate_content(
                model=MODEL_ID,
                contents=user_prompt,
                config=types.GenerateContentConfig(
                    system_instruction=system_prompt,
                    max_output_tokens=MAX_OUTPUT_TOKENS,
                    temperature=0.0,
                    thinking_config=types.ThinkingConfig(
                        thinking_budget=THINKING_BUDGET,
                    ),
                ),
            )
            elapsed = (time.time() - t0) / 60.0
            raw = ''
            for part in response.candidates[0].content.parts:
                if not getattr(part, 'thought', False):
                    raw += (part.text or '')
            return raw.strip(), elapsed

        except Exception as e:
            if attempt < max_retries - 1 and ('503' in str(e) or '429' in str(e)):
                print(f'  [Retry {attempt+1}/{max_retries}] {e} — waiting {wait}s')
                time.sleep(wait)
                wait *= 2
            else:
                raise


def parse_audit_response(raw: str) -> dict:
    """
    Parse the JSON audit response from Gemini.
    Returns a safe dict even if parsing fails.
    """
    try:
        clean = re.sub(r'```(?:json)?', '', raw).strip().rstrip('`').strip()
        match = re.search(r'\{.*\}', clean, re.DOTALL)
        if match:
            return json.loads(match.group(0))
    except Exception:
        pass
    return {'parse_error': True, 'raw_audit_response': raw[:500]}

## 6. Audit System Prompt

In [ ]:
AUDIT_SYSTEM_PROMPT = """\
You are auditing the output of an automated puzzle equivalence judge.
You will receive:
  1. An adapted puzzle
  2. The judge's verdict (JSON)
  3. The judge's think trace (may be empty)

Your job is to check four things and report only what you observe.
Do NOT re-judge equivalence yourself. Do NOT solve the puzzle.

---
CHECK 1 — PUZZLE STRUCTURE
Does the adapted puzzle contain all of the following?
  - <context> block
  - <clues> block
  - <question> block
  - Answer choices A through E

Set puzzle_structure_valid = true only if ALL are present.
Otherwise list what is missing in structure_issues.

---
CHECK 2 — JUDGE REACHED A DECISION
Look at the judge's think trace and verdict.
Did the judge actually reach a final logical_equivalence_confirmed value
(true or false), or did it fail before concluding (e.g. parse error,
empty verdict, judge_parse_exhausted)?

Set judge_reached_decision = true if a real true/false was produced.
Set judge_reached_decision = false if the verdict is an error or inconclusive.

---
CHECK 3 — NATURE OF JUDGE FAILURE (only if judge said FAIL)
If the judge returned format_check = FAIL, look at the feedback field.
Classify the failure:
  - "formatting_only"    : failures are purely structural (missing tags,
                           wrong choice count, leaked entity names)
  - "equivalence_issue"  : judge cited genuine logical differences
                           (wrong clue type, direction flipped, clue merged)
  - "mixed"              : both formatting and equivalence issues cited
  - "n/a"                : judge passed, or judge did not reach a decision

Set failure_nature to one of the four values above.

---
CHECK 4 — JUDGE'S FINAL DECISION
Report what logical_equivalence_confirmed value the judge produced.
  - true   : judge confirmed equivalent
  - false  : judge rejected
  - null   : judge did not reach a decision

---
OUTPUT JSON ONLY. No markdown, no explanation, no ```json blocks.

{
  "puzzle_structure_valid": true|false,
  "structure_issues": "<comma-separated list of missing elements or empty string>",
  "judge_reached_decision": true|false,
  "failure_nature": "formatting_only|equivalence_issue|mixed|n/a",
  "judge_final_decision": true|false|null,
  "audit_notes": "<one sentence summary of any notable issue, or empty string>"
}
"""

## 7. Run Audit

In [ ]:
KEY_COLS = ['Puzzle_Number', 'Question_Number']
audit_records, audit_done = load_completed(AUDIT_OUT, KEY_COLS)

for _, row in tqdm(audit_df.iterrows(), total=len(audit_df), desc='Audit'):
    key = (row['Puzzle_Number'], row['Question_Number'])
    if key in audit_done:
        continue

    user_prompt = f"""<adapted_puzzle>
{row['final_adapted_puzzle']}
</adapted_puzzle>

<judge_verdict>
{row['final_judge_verdict']}
</judge_verdict>

<judge_think_trace>
{row['final_judge_think_trace'] or '(none)'}
</judge_think_trace>"""

    raw_response = ''
    elapsed      = 0.0
    audit        = {}

    try:
        raw_response, elapsed = generate(AUDIT_SYSTEM_PROMPT, user_prompt)
        audit = parse_audit_response(raw_response)
    except Exception as e:
        print(f'  [P{row["Puzzle_Number"]} Q{row["Question_Number"]}] Error: {e}')
        raw_response = f'ERROR: {e}'
        audit = {'parse_error': True}

    audit_records.append({
        'Puzzle_Number':          row['Puzzle_Number'],
        'Question_Number':        row['Question_Number'],
        'final_outcome':          row['final_outcome'],
        'generator_model':        row.get('generator_model', ''),
        # Audit fields
        'puzzle_structure_valid': audit.get('puzzle_structure_valid'),
        'structure_issues':       audit.get('structure_issues', ''),
        'judge_reached_decision': audit.get('judge_reached_decision'),
        'failure_nature':         audit.get('failure_nature', ''),
        'judge_final_decision':   audit.get('judge_final_decision'),
        'audit_notes':            audit.get('audit_notes', ''),
        'audit_parse_error':      audit.get('parse_error', False),
        'raw_audit_response':     raw_response,
        'audit_elapsed_min':      round(elapsed, 3),
    })
    audit_done.add(key)

    if len(audit_records) % SAVE_EVERY == 0:
        save_results(audit_records, AUDIT_OUT)
        print(f'  [Checkpoint] {len(audit_records)} rows saved.')

save_results(audit_records, AUDIT_OUT)
audit_df_out = pd.DataFrame(audit_records)
print(f'\nAudit complete. {len(audit_df_out)} rows -> {AUDIT_OUT}')

## 8. Audit Summary

In [ ]:
audit_df_out = pd.read_csv(AUDIT_OUT)
n = len(audit_df_out)

print(f"Total audited            : {n}")
print(f"Structure valid          : {audit_df_out['puzzle_structure_valid'].eq(True).sum()}")
print(f"Structure invalid        : {audit_df_out['puzzle_structure_valid'].eq(False).sum()}")
print(f"Judge reached decision   : {audit_df_out['judge_reached_decision'].eq(True).sum()}")
print(f"Judge did not decide     : {audit_df_out['judge_reached_decision'].eq(False).sum()}")
print()
print("Failure nature breakdown:")
print(audit_df_out['failure_nature'].value_counts().to_string())
print()
print("Judge final decision breakdown:")
print(audit_df_out['judge_final_decision'].value_counts(dropna=False).to_string())
print()

# Rows safe to include in benchmark:
# structure valid AND (judge confirmed equivalent OR failure was formatting-only)
rescuable = audit_df_out[
    audit_df_out['puzzle_structure_valid'].eq(True) &
    audit_df_out['failure_nature'].isin(['formatting_only', 'n/a'])
]
print(f"Potentially rescuable (structure ok, no equivalence issues): {len(rescuable)}")

## 9. Export Clean List for Benchmark Pooling

In [ ]:
# Rows confirmed safe for benchmark:
# - structure valid
# - judge reached a decision
# - judge confirmed equivalent OR failure was formatting-only (rescuable)
clean = audit_df_out[
    audit_df_out['puzzle_structure_valid'].eq(True) &
    audit_df_out['judge_reached_decision'].eq(True) &
    (
        audit_df_out['judge_final_decision'].eq(True) |
        audit_df_out['failure_nature'].eq('formatting_only')
    )
][['Puzzle_Number', 'Question_Number', 'generator_model',
   'final_outcome', 'failure_nature', 'audit_notes']]

clean_path = os.path.join(WORKDIR, 'audit_clean_for_pool.csv')
clean.to_csv(clean_path, index=False)
print(f'Clean rows for pooling: {len(clean)} -> {clean_path}')
clean.head(10)